# Phase 5: FLEURS checks, multilingual and forgetting

Two questions, both on FLEURS test (read speech, CC-BY 4.0), with the same scoring code as the Cypriot runs:

**A. Did Reson8's compression hurt other languages?** Zero-shot `Qwen/Qwen3-ASR-0.6B-hf` vs `Reson8/Rumble-2-Flash`,
`N_A` clips per language. The language mix comes from `docs/tokenizer_diff.md`. It covers European languages,
languages whose script lost many tokens (Thai 62% removed, Hangul 32%, Arabic 21%), and CJK, where CER is the metric.

**B. Does Cypriot fine-tuning hurt Standard Modern Greek?** This is the forgetting check from the open items in
`docs/PROJECT_LOG.md`. FLEURS `el_gr` test, zero-shot vs our LoRA fine-tunes, for whisper-medium, Rumble and Qwen.
It also counts Cypriot forms leaking into Standard Greek output, such as τζαι for και and εν for είναι.

Caveats: FLEURS is read speech and differs from our spontaneous Cypriot audio, so compare models only on the same
FLEURS data. FLEURS references write numbers as digits and have no fillers. N_A = 100 clips per language is a
quick check, not a leaderboard number: differences under about 2 points WER are noise at this size.

Runtime: **L4** (bfloat16) or T4 (float32, about 2× slower). On a T4, part A took about 75 min and part B (200 clips) about 1 h.
Results of the first run: `docs/PROJECT_LOG.md` section 15.
FLEURS audio is streamed from the Hugging Face Hub, and only the needed clips are read.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
OUT = '/content/drive/MyDrive/cyasr/runs_fleurs'      # predictions + scores, survive disconnects
FT = '/content/drive/MyDrive/cyasr/runs'               # our fine-tuned checkpoints (…/best)

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print(transformers.__version__)

In [ ]:
# FLEURS loader: streams data/<lang>/audio/test.tar.gz and stops after n clips (no full download).
# TSV columns: id, file_name, raw_transcription, transcription, characters, num_samples, gender.
import csv, io, json, os, tarfile, time, gc
import numpy as np, requests, soundfile as sf, torch

FLEURS = 'https://huggingface.co/datasets/google/fleurs/resolve/main/data'

def load_fleurs(lang, n=None):
    tsv = requests.get(f'{FLEURS}/{lang}/test.tsv', timeout=60)
    tsv.raise_for_status()
    refs = {}
    for row in csv.reader(io.StringIO(tsv.text), delimiter='\t', quoting=csv.QUOTE_NONE):
        refs.setdefault(row[1], (row[0], row[2]))   # file_name -> (utterance id, raw transcription)
    clips = []
    with requests.get(f'{FLEURS}/{lang}/audio/test.tar.gz', stream=True, timeout=60) as r:
        r.raise_for_status()
        r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode='r|gz') as tar:
            for m in tar:
                name = os.path.basename(m.name)
                if not m.isfile() or name not in refs:
                    continue
                audio, sr = sf.read(io.BytesIO(tar.extractfile(m).read()), dtype='float32')
                assert sr == 16000, sr
                uid, ref = refs[name]
                clips.append({'audio_id': name, 'utt_id': uid, 'ref': ref, 'audio': audio,
                              'duration_s': len(audio) / sr})
                if n and len(clips) >= n:
                    break
    return clips

t0 = time.time(); demo = load_fleurs('el_gr', 3)
print(f'{time.time()-t0:.1f}s', [(c['audio_id'], round(c['duration_s'], 1), c['ref'][:60]) for c in demo])

In [ ]:
# Scoring: same corpus-level WER/CER as the Cypriot runs (cyasr.metrics.corpus_rate) on the 'basic' level
# (NFC, lowercase, punctuation removed). Languages without spaces are scored by CER with spaces removed.
from cyasr.metrics import corpus_rate
from cyasr.normalize import normalize

NO_SPACES = {'cmn_hans_cn', 'ja_jp', 'th_th'}

def score(lang, refs, hyps):
    r = [normalize(x, 'basic') for x in refs]
    h = [normalize(x, 'basic') for x in hyps]
    out = {'wer': corpus_rate(r, h, 'word')['rate'], 'cer': corpus_rate(r, h, 'char')['rate']}
    if lang in NO_SPACES:
        out['cer'] = corpus_rate([x.replace(' ', '') for x in r], [x.replace(' ', '') for x in h], 'char')['rate']
        out['wer'] = None
    out['empty'] = sum(not x.strip() for x in hyps)
    return out

def save(run, lang, clips, hyps, extra=None):
    d = f'{OUT}/{run}_{lang}'; os.makedirs(d, exist_ok=True)
    with open(f'{d}/predictions.jsonl', 'w', encoding='utf-8') as f:
        for c, h in zip(clips, hyps):
            f.write(json.dumps({'audio_id': c['audio_id'], 'utt_id': c['utt_id'], 'duration_s': c['duration_s'],
                                'ref': c['ref'], 'hyp': h}, ensure_ascii=False) + '\n')
    s = score(lang, [c['ref'] for c in clips], hyps) | {'n': len(clips)} | (extra or {})
    json.dump(s, open(f'{d}/metrics.json', 'w'), indent=1)
    return s

def done(run, lang):
    return os.path.exists(f'{OUT}/{run}_{lang}/metrics.json')

def free_gpu():   # call after `t = None` so the model has no references left
    gc.collect(); torch.cuda.empty_cache()

## A. Zero-shot Qwen3-ASR-0.6B vs Rumble-2-Flash on 12 languages

Both run with the language forced (ISO code, as in our configs), greedy decoding, batch 1, and pinned revisions.
Rumble uses its product defaults (`template_kwargs: {}`): digits on and fillers off, which matches FLEURS references.
The smoke test prints a few outputs per language first. If a language code is rejected or the output is in the wrong
language, fix `LANGS` before the full run.

In [ ]:
from cyasr.transcribe_qwen import QwenTranscriber

# FLEURS config -> language code passed to the model (ISO 639-1, as our configs pass "el")
LANGS = {'el_gr': 'el', 'nl_nl': 'nl', 'de_de': 'de', 'pl_pl': 'pl', 'ru_ru': 'ru', 'ar_eg': 'ar',
         'hi_in': 'hi', 'th_th': 'th', 'cmn_hans_cn': 'zh', 'ja_jp': 'ja', 'ko_kr': 'ko', 'vi_vn': 'vi'}
N_A = 100
MODELS_A = {
    'qwen3-asr-0.6b': dict(model_name='Qwen/Qwen3-ASR-0.6B-hf', revision='7f1569a48a89f3e3f4dc3a5c9d28bddd903bc76c'),
    'rumble-2-flash': dict(model_name='Reson8/Rumble-2-Flash', revision='95bd5f9133a2398e5e48a98c089fe48e122bde97'),
}
DATA_A = {}
for lang in LANGS:
    t0 = time.time(); DATA_A[lang] = load_fleurs(lang, N_A)
    print(f'{lang}: {len(DATA_A[lang])} clips, {sum(c["duration_s"] for c in DATA_A[lang])/60:.1f} min audio, {time.time()-t0:.0f}s')

In [ ]:
# Smoke test: 2 clips per language per model
for run, kw in MODELS_A.items():
    t = QwenTranscriber(**kw, language='el', batch_size=1, max_new_tokens=1024)
    for lang, code in LANGS.items():
        t.language = code   # the transcriber reads self.language on every call
        hyps = t.transcribe([c['audio'] for c in DATA_A[lang][:2]])
        print(f'--- {run} {lang}\nREF: {DATA_A[lang][0]["ref"][:120]}\nHYP: {hyps[0][:120]}')
    t = None; free_gpu()

In [ ]:
# Full part A. Resumable: finished (model, language) pairs on Drive are skipped.
for run, kw in MODELS_A.items():
    t = None
    for lang, code in LANGS.items():
        if done(run, lang):
            continue
        if t is None:
            t = QwenTranscriber(**kw, language=code, batch_size=1, max_new_tokens=1024)
        t.language = code
        t0 = time.time()
        hyps = t.transcribe([c['audio'] for c in DATA_A[lang]])
        s = save(run, lang, DATA_A[lang], hyps, {'seconds': time.time() - t0})
        print(run, lang, s)
    t = None; free_gpu()

In [ ]:
def show_a():
    print(f"{'language':13} {'Qwen WER':>9} {'Rumble WER':>10} {'Qwen CER':>9} {'Rumble CER':>10} {'Δ CER':>7}  {'Qwen s':>6} {'Rumble s':>8}")
    for lang in LANGS:
        q = json.load(open(f'{OUT}/qwen3-asr-0.6b_{lang}/metrics.json'))
        r = json.load(open(f'{OUT}/rumble-2-flash_{lang}/metrics.json'))
        f = lambda x: '   -' if x is None else f'{x:.1%}'
        print(f"{lang:13} {f(q['wer']):>9} {f(r['wer']):>10} {q['cer']:9.1%} {r['cer']:10.1%} {r['cer']-q['cer']:+7.1%}  "
              f"{q['seconds']:6.0f} {r['seconds']:8.0f}")
show_a()

## B. Forgetting on Standard Modern Greek (FLEURS `el_gr`, full test set)

Six systems: zero-shot vs our LoRA fine-tune, for whisper-medium, Rumble and Qwen. Each is decoded exactly as in
its Cypriot evaluation config:
- whisper-medium: fallback decoding, no-speech threshold 0.6, batch 8.
- Rumble: `{digits: false, filler: true, verbatim: true}`. Its LoRA was trained with these options, so the zero-shot
  run uses them too, to keep the prompt identical.
- Qwen: its plain prompt.

FLEURS references use digits, so Rumble's `digits: false` costs it a little on numbers in both runs alike. That
doesn't affect the zero-shot vs fine-tuned difference.

In [ ]:
from cyasr.transcribe import Transcriber

RUMBLE_KW = {'digits': False, 'filler': True, 'verbatim': True}
SYSTEMS_B = {
    'whisper-medium':      lambda: Transcriber('openai/whisper-medium', 'el', 'transcribe', 1, 'sequential', 8, 'fallback', 0.6),
    'whisper-medium_lora': lambda: Transcriber(f'{FT}/finetune_whisper-medium_lora/best', 'el', 'transcribe', 1, 'sequential', 8, 'fallback', 0.6),
    'rumble-2-flash_m':    lambda: QwenTranscriber('Reson8/Rumble-2-Flash', '95bd5f9133a2398e5e48a98c089fe48e122bde97', 'el', 1, 1024, 'auto', RUMBLE_KW),
    'rumble-2-flash_lora': lambda: QwenTranscriber(f'{FT}/finetune_rumble-2-flash_lora/best', None, 'el', 1, 1024, 'auto', RUMBLE_KW),
    'qwen3-asr-0.6b':      lambda: QwenTranscriber('Qwen/Qwen3-ASR-0.6B-hf', '7f1569a48a89f3e3f4dc3a5c9d28bddd903bc76c', 'el', 1, 1024, 'auto', {}),
    'qwen3-asr-0.6b_lora': lambda: QwenTranscriber(f'{FT}/finetune_qwen3-asr-0.6b_lora/best', None, 'el', 1, 1024, 'auto', {}),
}
for name in SYSTEMS_B:
    if 'lora' in name:
        path = f"{FT}/finetune_{name.replace('_lora', '')}_lora/best"
        print(name, 'OK' if os.path.isdir(path) else f'MISSING {path}')

N_B = 200       # first 200 el_gr test clips (~1 h on a T4 for all six systems); None = full test set (~3.5 h on a T4)
DATA_A = None; gc.collect()   # free part A audio: the kernel ran out of RAM (12 GB) loading a float32 model with it in memory
GREEK = load_fleurs('el_gr', N_B)
print(len(GREEK), 'clips', f"{sum(c['duration_s'] for c in GREEK)/60:.0f} min")

In [ ]:
import re
from collections import Counter
# Cypriot forms (nodiac spelling) that should be rare in Standard Greek output; και / είναι are their standard forms.
MARKERS = ['τζαι', 'τζ', 'εν', 'εννα', 'ουλλοι', 'ουλλα', 'τουτο', 'τουτη', 'ποττε', 'και', 'ειναι']

def marker_counts(texts):
    c = Counter(w for t in texts for w in normalize(t, 'nodiac').split())
    return {m: c[m] for m in MARKERS}

for name, make in SYSTEMS_B.items():
    if done(name, 'el_gr'):
        continue
    t0 = time.time(); t = make()
    hyps = t.transcribe([c['audio'] for c in GREEK])
    s = save(name, 'el_gr', GREEK, hyps, {'seconds': time.time() - t0, 'markers': marker_counts(hyps)})
    print(name, {k: v for k, v in s.items() if k != 'markers'})
    t = None; free_gpu()

In [ ]:
def show_b():
    ref = marker_counts([c['ref'] for c in GREEK])
    print(f"{'system':22} {'WER basic':>9} {'CER':>6}  " + ' '.join(f'{m:>6}' for m in MARKERS))
    print(f"{'(reference)':22} {'':>9} {'':>6}  " + ' '.join(f'{ref[m]:6}' for m in MARKERS))
    for name in SYSTEMS_B:
        m = json.load(open(f'{OUT}/{name}_el_gr/metrics.json'))
        print(f"{name:22} {m['wer']:9.1%} {m['cer']:6.1%}  " + ' '.join(f"{m['markers'][k]:6}" for k in MARKERS))
show_b()

In [ ]:
# Paired bootstrap on clip-level errors: fine-tuned minus zero-shot WER, 95% CI (same method as compare_finetuned.py).
from cyasr.metrics import error_counts
def preds(name):
    return [json.loads(l) for l in open(f'{OUT}/{name}_el_gr/predictions.jsonl', encoding='utf-8')]
def bootstrap(a, b, n=10000, seed=42):
    ea = np.array([error_counts(normalize(r['ref'], 'basic'), normalize(r['hyp'], 'basic'))[0] for r in a])
    eb = np.array([error_counts(normalize(r['ref'], 'basic'), normalize(r['hyp'], 'basic'))[0] for r in b])
    nw = np.array([len(normalize(r['ref'], 'basic').split()) for r in a])
    rng = np.random.default_rng(seed); idx = rng.integers(0, len(a), (n, len(a)))
    d = (eb[idx].sum(1) - ea[idx].sum(1)) / nw[idx].sum(1)
    return (eb.sum() - ea.sum()) / nw.sum(), np.percentile(d, [2.5, 97.5])
for base, ft in [('whisper-medium', 'whisper-medium_lora'), ('rumble-2-flash_m', 'rumble-2-flash_lora'),
                 ('qwen3-asr-0.6b', 'qwen3-asr-0.6b_lora')]:
    d, ci = bootstrap(preds(base), preds(ft))
    print(f'{ft:22} vs zero-shot: {d:+.1%}  95% CI [{ci[0]:+.1%}, {ci[1]:+.1%}]   (positive = worse after fine-tuning)')

In [ ]:
# Release the GPU when done (Colab bills idle runtimes too)
from google.colab import runtime
runtime.unassign()